In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import Window
import snowflake.snowpark.functions as F

session = get_active_session()

In [ ]:
df_token = session.table("BIGDATA_DB.RAW.TOKENS")
df_token.show()
df_token.print_schema()
print(f"\nTotal token:{df_token.count()}")

In [ ]:
df_token_processed = (
    df_token
    .filter(F.col("ADDRESS").is_not_null() & (F.col("SYMBOL") != '') & (F.col("NAME") != ''))
    .filter(F.col("DECIMALS") <= 36)
    .drop_duplicates("ADDRESS")
)

In [ ]:
df_token_processed.show()

In [ ]:
window_spec = Window.order_by("token_address")
df_map_token = (df_token_processed
                .select(F.col("address").alias("token_address"))
                .distinct()
                .with_column("token_id", F.row_number().over(window_spec))           
)
df_map_token.write.mode("overwrite").save_as_table(
    "BIGDATA_DB.STAGINg.MAP_TOKEN",
    table_type = "transient"
)


In [ ]:
df_map_token.show(10)

In [ ]:
df_map = session.table("BIGDATA_DB.STAGING.MAP_TOKEN")

df_indexed = df_token_processed.join(
    df_map_token,
    df_token_processed["address"] == df_map_token["token_address"],
    "inner"
)

df_final_indexed = df_indexed.select(
    F.col("token_id"),
    F.col("address"),
    F.col("name"),
    F.col("symbol"),
    F.col("decimals"),
    F.col("total_supply")
)

df_final_indexed.write.mode("overwrite").save_as_table(
    "BIGDATA_DB.STAGING.DIM_TOKENS_INDEXED", 
    table_type="transient"
)
df_final_indexed.show()